# Chapter 11 &mdash; NPDA and DPDA Are Different

**Concept 17 of the Chapter 11 decomposition:** *Combating Inherent Ambiguity: NPDA and DPDA Are Different*

Inherent ambiguity means no deterministic parser &mdash; unlike NFA and DFA, nondeterminism <i>adds power</i> here.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-NPDA-Vs-DPDA/Concept-NPDA-Vs-DPDA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


For finite automata, nondeterminism is a **convenience**: NFA and DFA recognise the
same class (Chapter 7). For pushdown automata it is **not**.

* **NPDA** recognise exactly the context-free languages.
* **DPDA** recognise a strictly smaller class, the **deterministic** CFLs.

The separation is visible from ambiguity. A DPDA has one computation per input, so it
induces at most one parse &mdash; roughly, an unambiguous grammar. An **inherently
ambiguous** language (Concept 11) therefore has no DPDA.

A simpler witness is $L_{pal} = \{ww^R\}$: a nondeterministic machine *guesses* the
midpoint; a deterministic one cannot know where it is.

This is why real parser generators restrict themselves to deterministic subclasses
(LL, LR) rather than accepting all CFGs.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### Even-length palindromes: the guessing language

In [ ]:
Pal = mkg({'S': ["", "aSa", "bSb"]})
print("L(Pal) :", language(Pal, 6)[:10], "...")

### And a language a deterministic machine handles easily

In [ ]:
Marked = mkg({'S': ["m", "aSa", "bSb"]})     # an explicit midpoint marker
print("L(Marked) :", language(Marked, 5)[:8], "...")

## 3. Tests

Both are context-free.

In [ ]:
assert all(w == w[::-1] for w in language(Pal, 6))
assert all(w == w[::-1] for w in language(Marked, 5))
print("Pal    : even-length palindromes")
print("Marked : palindromes with an explicit centre marker 'm'")

**The difference is the midpoint.** With a marker, no guessing is needed.

In [ ]:
def det_check(s):
    # a DETERMINISTIC one-pass check, possible only because of the marker
    if s.count('m') != 1: return False
    i = s.index('m')
    return s[:i] == s[i+1:][::-1]

from itertools import product
L = set(language(Marked, 5))
want = {''.join(p) for k in range(6) for p in product('abm', repeat=k)
        if det_check(''.join(p))}
print("deterministic check agrees with the grammar :", L == want)
assert L == want

Without the marker, a single left-to-right pass has nothing to key on.

In [ ]:
print("reading 'aabbaa' left to right, when do you start matching backwards?")
for i in range(1, 6):
    s = 'aabbaa'
    print("   guess midpoint at %d : %-8s vs %-8s  match? %s"
          % (i, s[:i], s[i:][::-1], s[:i] == s[i:][::-1]))
print("\nOnly one guess works, and you cannot tell which without lookahead.")
print("An NPDA tries all of them at once.  A DPDA cannot.")

Ambiguity is the grammar-side symptom.

In [ ]:
Lamb = mkg({'S': ["XC", "AY"], 'X': ["", "aXb"], 'C': ["", "cC"],
            'A': ["", "aA"], 'Y': ["", "bYc"]})
print("inherently ambiguous language, parses of 'abc' :", nparses(Lamb, 'abc'))
assert nparses(Lamb, 'abc') >= 2
print("\nTwo parses means two accepting computations, which a DPDA cannot have.")

Why it matters in practice.

In [ ]:
print("NFA -> DFA : subset construction, always possible (Chapter 7)")
print("NPDA -> DPDA : NO such construction exists")
print()
print("So parser generators accept LL(k) or LR(k) grammars -- deterministic")
print("subclasses -- and reject the rest, rather than determinizing them.")

## 4. Exercises


1. Give a CFL that is unambiguous but still has no DPDA. (Hint: prefix property.)
2. Why does a DPDA's single computation limit it to one parse?
3. Which is bigger: the LR(1) languages or the deterministic CFLs?

In [ ]:
# Your work for the exercises above.